# Thử nghiệm trên dump có sẵn: hiệu chỉnh logit KAC và giảm chiều Spherical (4 dataset, Val, 3 seed)

Không train, không cần GPU hay Internet: chỉ đọc dump của các phiên APT đã chạy (logit KAC + đặc trưng ViT đóng băng).

**Thử nghiệm 1 — hiệu chỉnh logit KAC theo khối task.** KAC học bằng CE trong từng task nên mỗi khối logit có độ lệch riêng, argmax trên mọi lớp hay chọn sai task. Thử các phép biến đổi không cần học (`center`, `center_half`, `zscore`, `block_softmax`): 0 tham số, 0 nhánh phụ, vẫn 1 lượt forward. Sau đó ghép KAC đã hiệu chỉnh với Spherical-ViT (fuse/hier) xem còn tăng không.

**Thử nghiệm 2 — giảm chiều đặc trưng Spherical** 768 → d ∈ {64, 128, 256} bằng PCA (học từ ảnh train task 1 rồi đóng băng) hoặc chiếu ngẫu nhiên trực giao. So với bản đầy đủ 768 chiều trên cùng seed.

**Kaggle:** Accelerator **None** (CPU, không tốn quota GPU). Add Input output các notebook APT chứa `runs/kac/<dataset>/val/dumps/seed*` (giống notebook Swin; không cần dataset ảnh). Run All.


## 1. Cấu hình

In [1]:
# ==============================================================================
# 1. CẤU HÌNH  (chỉ đọc dump có sẵn -> chạy CPU, không cần GPU / Internet)
# ==============================================================================
DATASETS = ["CIFAR100", "CUB200", "StanfordCars", "ImageNet_R"]
SEEDS    = [0, 1, 2]
PHASE    = "val"
INPUT_ROOT = "/kaggle/input"
OUT = "/kaggle/working/exp_calib_proj"
# Thử nghiệm 1: hiệu chỉnh logit KAC theo từng khối task (0 tham số, 0 nhánh phụ)
CALIB = ["raw", "center", "center_half", "zscore", "block_softmax"]
CALIB_FUSE = ["center", "block_softmax"]      # các cách hiệu chỉnh đem ghép tiếp với Spherical-ViT (fuse/hier)
# Thử nghiệm 2: giảm chiều đặc trưng Spherical (ViT đóng băng) 768 -> d
PROJ_DIMS  = [64, 128, 256]
PROJ_TYPES = ["pca", "rp"]                    # pca: học từ ảnh train task 1 rồi đóng băng; rp: chiếu ngẫu nhiên trực giao, không cần dữ liệu
import os; os.makedirs(OUT, exist_ok=True)


## 2. Thư viện phân tích

In [2]:
# ==============================================================================
# 2. THƯ VIỆN PHÂN TÍCH (y nguyên code Spherical đã dùng)
# ==============================================================================
"""Offline analysis of the dumps written by the patched APT repo (apt_patch.py).
Per seed dir:  train_frozen_{t}.pt {f,y}   test_frozen_{j}.pt {f,y}   test_{t}_{j}.pt {k,y}  (k = logits over the t+1 seen tasks).
Labels are already in task order (class c of task j lies in [lo_j, hi_j)), frozen/KAC rows of a test set are in the same order."""
import os, glob, math
import numpy as np
import torch
import torch.nn.functional as F

W_GRID = [0.25, 0.5, 0.75, 1.0]
KAPPA_GRID = [8, 16, 32, 64]
K_GRID = [1, 3, 5]
LAM_GRID = [0.5, 1.0, 2.0]
WIN_GRID = [0.0, 0.5]


def embed(f):
    return F.normalize(F.layer_norm(f.float(), (f.shape[1],)), dim=1)


def sph_kmeans(Z, k, g, iters=20):
    k = min(k, Z.shape[0])
    C = Z[torch.randperm(Z.shape[0], generator=g)[:k]].clone()
    for _ in range(iters):
        a = (Z @ C.T).argmax(1)
        newC = torch.zeros_like(C).index_add_(0, a, Z)
        empty = newC.norm(dim=1) < 1e-8
        if empty.any():
            newC[empty] = Z[torch.randint(0, Z.shape[0], (int(empty.sum()),), generator=g)]
        C = F.normalize(newC, dim=1)
    return C


def seeds_complete(run_dir):
    res = {}
    for d in sorted(glob.glob(os.path.join(run_dir, "seed*"))):
        T = len(glob.glob(os.path.join(d, "train_frozen_*.pt")))
        if T and all(os.path.exists(os.path.join(d, f"test_{t}_{j}.pt")) for t in range(T) for j in range(t + 1)) \
           and all(os.path.exists(os.path.join(d, f"test_frozen_{j}.pt")) for j in range(T)):
            res[int(os.path.basename(d)[4:])] = (d, T)
    return res


def block_lsm(x, bounds):
    out = torch.empty_like(x)
    for lo, hi in bounds:
        out[:, lo:hi] = F.log_softmax(x[:, lo:hi], 1)
    return out


def block_lse(ls, bounds):
    out = torch.empty_like(ls)
    for lo, hi in bounds:
        out[:, lo:hi] = torch.logsumexp(ls[:, lo:hi], 1, keepdim=True)
    return out


def _metrics(A, T):
    at = [A[:t + 1, t].mean() for t in range(T)]
    forg = np.mean([A[j, j] - A[j, T - 1] for j in range(T - 1)]) if T > 1 else 0.0   # = repo's calc_general_forgetting
    return float(np.mean(at)), float(at[-1]), float(forg)


def analyse_seed(d, T, seed):
    ld = lambda n: torch.load(os.path.join(d, n))
    tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
    te = [ld(f"test_frozen_{j}.pt") for j in range(T)]
    # ranh giới task lấy từ độ rộng logit (repo CUB có nhãn 1..200 nên 1 lớp rỗng, task chứa nó có 19 lớp nhưng vẫn chiếm 20 cột)
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    has = torch.zeros(C, dtype=torch.bool); has[ytr.unique()] = True
    print(f"    {int((~has).sum())} lớp không có ảnh train (bỏ qua):", (~has).nonzero().flatten().tolist())
    z0 = Ztr[:512]
    G = 1 - z0 @ z0.T
    inv_med = 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()
    g = torch.Generator().manual_seed(seed)
    cen = {k: torch.zeros(C, k, Ztr.shape[1]) for k in K_GRID}
    for c in range(C):
        if not has[c]: continue
        Zc = Ztr[ytr == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g)
            cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    acc, tid = {}, np.zeros((T, T))
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te[j]["y"].long()), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = embed(te[j]["f"]); lo, hi = bounds_all[j]
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    return rows, dict(seed=seed, task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"))


def analyse(run_dir):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        print(f"  seed {seed}: {T} task", flush=True)
        r, dg = analyse_seed(d, T, seed); rows += r; diag.append(dg)
    return rows, diag


"""Spherical branch on three feature sources, same KAC logits, same grids (needs fusion_online.py code in the same namespace).
  frozen      : ViT without prompt. Centres = train images of each task (once). Test features = frozen features.
  apt_stale   : APT features (CLS after the prompts). Centre of a class = APT features of its train images taken right after
                its own task and never refreshed; test features = APT features of the CURRENT model (what a real continual run has).
  apt_refresh : at every step, centres rebuilt from the current model's features of ALL seen train images.
                A real continual run does not keep old train images -> reference only (separates drift from feature quality).
Dumps: train_frozen_{t}.pt, train_apt_{t}_{j}.pt, test_frozen_{j}.pt, test_apt_{t}_{j}.pt, test_{t}_{j}.pt (see apt_patch.py)."""

SOURCES = ["frozen", "apt_stale", "apt_refresh"]


def _build_cen(Z, y, C, g):
    has = torch.zeros(C, dtype=torch.bool); has[y.unique()] = True
    cen = {k: torch.zeros(C, k, Z.shape[1]) for k in K_GRID}
    for c in has.nonzero().flatten().tolist():
        Zc = Z[y == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g); cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    return cen, has


def _inv_med(Z):
    z0 = Z[:512]; G = 1 - z0 @ z0.T
    return 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()


def sources_available(d, T):
    ex = lambda n: os.path.exists(os.path.join(d, n))
    out = ["frozen"]
    if all(ex(f"train_apt_{t}_{t}.pt") for t in range(T)) and all(ex(f"test_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)):
        out.append("apt_stale")
        if all(ex(f"train_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)): out.append("apt_refresh")
    return out


def analyse_seed_src(d, T, seed, source):
    ld = lambda n: torch.load(os.path.join(d, n))
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    te_y = [ld(f"test_frozen_{j}.pt")["y"].long() for j in range(T)]
    g = torch.Generator().manual_seed(seed)
    if source == "frozen":
        tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
        zte_cache = [embed(ld(f"test_frozen_{j}.pt")["f"]) for j in range(T)]
        zte = lambda t, j: zte_cache[j]
    else:
        tr = [ld(f"train_apt_{t}_{t}.pt") for t in range(T)]
        zte = lambda t, j: embed(ld(f"test_apt_{t}_{j}.pt")["f"])
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    if source != "apt_refresh":                                   # centres built once, from each task's own train images
        Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
        cen_s, has_s = _build_cen(Ztr, ytr, C, g); inv_s = _inv_med(Ztr)
        if source == "apt_stale":
            fz = [ld(f"train_frozen_{t}.pt")["y"] for t in range(T)]
            assert all(torch.equal(a["y"], b) for a, b in zip(tr, fz)), "train_apt và train_frozen lệch thứ tự"
    acc, tid = {}, np.zeros((T, T)); drift = np.full((T, T), np.nan)
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        if source == "apt_refresh":                               # re-embed everything seen so far with the current model
            parts = [ld(f"train_apt_{t}_{j}.pt") for j in range(t + 1)]
            Zt = torch.cat([embed(x["f"]) for x in parts]); yt = torch.cat([x["y"] for x in parts]).long()
            cen, has = _build_cen(Zt, yt, V, g); inv_med = _inv_med(embed(parts[0]["f"]))
        else:
            cen, has, inv_med = cen_s, has_s, inv_s
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te_y[j]), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = zte(t, j); lo, hi = bounds_all[j]
            if source == "apt_stale" and j < t:                   # how far did the same test images move since task j?
                drift[j, t] = (z * zte(j, j)).sum(1).mean().item()
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, source=source, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    dg = dict(seed=seed, source=source,
              task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"),
              cos_same_image_old_task_last=float(np.nanmean([drift[j, T - 1] for j in range(T - 1)])) if (source == "apt_stale" and T > 1) else float("nan"))
    return rows, dg


def analyse_all(run_dir, sources=SOURCES):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        have = sources_available(d, T)
        for s in sources:
            if s not in have:
                print(f"  seed {seed}: thiếu dump cho nguồn {s} (cần APT_DUMP_APT={'2' if s == 'apt_refresh' else '1'}) - bỏ qua"); continue
            print(f"  seed {seed}: {T} task, nguồn {s}", flush=True)
            r, dg = analyse_seed_src(d, T, seed, s); rows += r; diag.append(dg)
    return rows, diag


## 3. Tìm dump

In [3]:
# ==============================================================================
# 3. TÌM DUMP (output các notebook APT đã Add Input)
# ==============================================================================
import shutil, time, glob
import pandas as pd

def find_dumps(ds, seed):
    for d in sorted(glob.glob(f"{INPUT_ROOT}/**/runs/kac/{ds}/{PHASE}/dumps/seed{seed}", recursive=True)):
        T = len(glob.glob(f"{d}/train_frozen_*.pt"))
        if T and all(os.path.exists(f"{d}/test_{t}_{j}.pt") for t in range(T) for j in range(t + 1)) \
             and all(os.path.exists(f"{d}/test_frozen_{j}.pt") for j in range(T)):
            return d, T
    return None, None

RUNS = {}
for ds in DATASETS:
    for seed in SEEDS:
        d, T = find_dumps(ds, seed)
        print(f"{ds:13} seed {seed}: {d or 'KHÔNG CÓ DUMP -> bỏ qua'}")
        if d: RUNS[(ds, seed)] = (d, T)
assert RUNS, "Không thấy dump nào: Add Input output các notebook APT (chứa runs/kac/<dataset>/val/dumps/seed*)"

def bounds_of(d, T):
    Vs = [torch.load(f"{d}/test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    return [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]

def link_dir(dst, src, names):
    shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    for n in names: os.symlink(f"{src}/{n}", f"{dst}/{n}")


CIFAR100      seed 0: /kaggle/input/datasets/thnhtranhuynh/cifar-seed0-dumps/apt_spherical/runs/kac/CIFAR100/val/dumps/seed0
CIFAR100      seed 1: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cifar/apt_spherical/runs/kac/CIFAR100/val/dumps/seed1
CIFAR100      seed 2: /kaggle/input/notebooks/meomeomeo124/fork-of-apt-kac-spherical-cifar/apt_spherical/runs/kac/CIFAR100/val/dumps/seed2
CUB200        seed 0: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cub/apt_spherical/runs/kac/CUB200/val/dumps/seed0
CUB200        seed 1: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cub/apt_spherical/runs/kac/CUB200/val/dumps/seed1
CUB200        seed 2: /kaggle/input/notebooks/thnhtranhuynh/spherical-kac-cub/apt_spherical/runs/kac/CUB200/val/dumps/seed2
StanfordCars  seed 0: /kaggle/input/notebooks/huynhthinhatran/spherical-kac-cars/apt_spherical/runs/kac/StanfordCars/val/dumps/seed0
StanfordCars  seed 1: /kaggle/input/notebooks/huynhthinhatran/spherical-kac-cars/apt_spherical/runs/kac

## 4. Thử nghiệm 1: hiệu chỉnh logit KAC

In [4]:
# ==============================================================================
# 4. THỬ NGHIỆM 1: HIỆU CHỈNH LOGIT KAC THEO KHỐI TASK
#    KAC học bằng CE trong từng task -> mỗi khối logit có độ lệch/thang riêng -> argmax trên mọi lớp hay chọn sai task.
#    Các cách sau chỉ biến đổi logit của CHÍNH ảnh đó (không học gì, không dùng nhãn, không thêm nhánh):
#      raw           : argmax như KAC gốc
#      center        : trừ trung bình logit trong từng khối         (bỏ độ lệch của khối)
#      center_half   : trừ 0.5 x trung bình khối                     (bỏ một nửa)
#      zscore        : (logit - trung bình khối) / độ lệch chuẩn khối (bỏ cả lệch lẫn thang)
#      block_softmax : log-softmax trong từng khối                   (chọn khối tự tin nhất)
# ==============================================================================
def calibrate(k, bounds, how):
    if how == "raw": return k
    out = torch.empty_like(k)
    for lo, hi in bounds:
        s = k[:, lo:hi]
        if how == "center":        out[:, lo:hi] = s - s.mean(1, keepdim=True)
        elif how == "center_half": out[:, lo:hi] = s - 0.5 * s.mean(1, keepdim=True)
        elif how == "zscore":      out[:, lo:hi] = (s - s.mean(1, keepdim=True)) / (s.std(1, keepdim=True) + 1e-6)
        elif how == "block_softmax": out[:, lo:hi] = F.log_softmax(s, 1)
        else: raise ValueError(how)
    return out

ROWS1, ROWS1F = [], []
t_all = time.time()
for (ds, seed), (d, T) in RUNS.items():
    bounds = bounds_of(d, T)
    acc = {h: np.zeros((T, T)) for h in CALIB}; tid = {h: np.zeros((T, T)) for h in CALIB}
    for t in range(T):
        bt = bounds[:t + 1]
        for j in range(t + 1):
            v = torch.load(f"{d}/test_{t}_{j}.pt"); k, y = v["k"].float(), v["y"].long(); lo, hi = bounds[j]
            for h in CALIB:
                p = calibrate(k, bt, h).argmax(1)
                acc[h][j, t] = 100.0 * (p == y).float().mean().item()
                tid[h][j, t] = 100.0 * ((p >= lo) & (p < hi)).float().mean().item()
    for h in CALIB:
        avg, last, forg = _metrics(acc[h], T)
        ROWS1.append(dict(dataset=ds, seed=seed, calib=h, avg=avg, last=last, forgetting=forg,
                          task_id_last=float(np.mean([tid[h][j, T - 1] for j in range(T)]))))
    # ghép KAC đã hiệu chỉnh với Spherical-ViT đóng băng (fuse / hier), cùng lưới tham số như trước
    for h in CALIB_FUSE:
        w = f"{OUT}/work/calib/{ds}/seed{seed}/{h}"
        link_dir(w, d, [f"train_frozen_{t}.pt" for t in range(T)] + [f"test_frozen_{j}.pt" for j in range(T)])
        for t in range(T):
            for j in range(t + 1):
                v = torch.load(f"{d}/test_{t}_{j}.pt")
                torch.save({"k": calibrate(v["k"].float(), bounds[:t + 1], h), "y": v["y"]}, f"{w}/test_{t}_{j}.pt")
        r, _ = analyse_seed_src(w, T, seed, "frozen")
        for x in r: x.update(dataset=ds, kac=h)
        ROWS1F += [x for x in r if x["variant"] in ("fuse", "hier")]
    print(f"{ds:13} seed {seed}: xong ({time.time() - t_all:.0f}s)", flush=True)
pd.DataFrame(ROWS1).to_csv(f"{OUT}/exp1_calib_kac_rows.csv", index=False)
pd.DataFrame(ROWS1F).to_csv(f"{OUT}/exp1_calib_fuse_rows.csv", index=False)


CIFAR100      seed 0: xong (92s)
CIFAR100      seed 1: xong (123s)
CIFAR100      seed 2: xong (151s)
CUB200        seed 0: xong (172s)
CUB200        seed 1: xong (192s)
CUB200        seed 2: xong (211s)
StanfordCars  seed 0: xong (229s)
StanfordCars  seed 1: xong (247s)
StanfordCars  seed 2: xong (264s)
ImageNet_R    seed 0: xong (341s)
ImageNet_R    seed 1: xong (367s)
ImageNet_R    seed 2: xong (392s)


## 5. Thử nghiệm 2: giảm chiều Spherical

In [5]:
# ==============================================================================
# 5. THỬ NGHIỆM 2: GIẢM CHIỀU ĐẶC TRƯNG SPHERICAL (ViT đóng băng) 768 -> d
#    z0 = LayerNorm(f); p = (z0 - μ) W  (W: 768 x d); z = p / ||p||  -> spherical k-means, vMF như cũ
#    pca : μ, W học từ ảnh train của TASK 1 rồi giữ nguyên (không nhìn dữ liệu task sau)
#    rp  : μ = 0, W = ma trận trực giao ngẫu nhiên (seed cố định), không cần dữ liệu
#    "full" (d = 768, không chiếu) chính là Spherical-ViT đã có -> chạy lại để so cặp.
# ==============================================================================
_embed_orig = embed
def ln(f): return F.layer_norm(f.float(), (f.shape[1],))

def proj_matrix(kind, d, Z0_task1, seed):
    D = Z0_task1.shape[1]
    if kind == "pca":
        mu = Z0_task1.mean(0)
        U, S, Vh = torch.linalg.svd(Z0_task1 - mu, full_matrices=False)
        return mu, Vh[:d].T.contiguous(), float((S[:d] ** 2).sum() / (S ** 2).sum())
    g = torch.Generator().manual_seed(1000 + seed)
    Q, _ = torch.linalg.qr(torch.randn(D, d, generator=g))
    return torch.zeros(D), Q, float("nan")

ROWS2, EVR = [], []
t_all = time.time()
for (ds, seed), (d, T) in RUNS.items():
    tr = [torch.load(f"{d}/train_frozen_{t}.pt") for t in range(T)]
    te = [torch.load(f"{d}/test_frozen_{j}.pt") for j in range(T)]
    kac_files = [f"test_{t}_{j}.pt" for t in range(T) for j in range(t + 1)]
    # full = Spherical-ViT gốc
    globals()["embed"] = _embed_orig
    r, _ = analyse_seed_src(d, T, seed, "frozen")
    for x in r: x.update(dataset=ds, proj="full", dim=768)
    ROWS2 += [x for x in r if x["variant"] in ("fuse", "hier")]
    for kind in PROJ_TYPES:
        for dim in PROJ_DIMS:
            mu, W, evr = proj_matrix(kind, dim, ln(tr[0]["f"]), seed)
            EVR.append(dict(dataset=ds, seed=seed, proj=kind, dim=dim, explained_var_task1=evr))
            w = f"{OUT}/work/proj/{ds}/seed{seed}/{kind}{dim}"
            link_dir(w, d, kac_files)
            for t in range(T):
                torch.save({"f": (ln(tr[t]["f"]) - mu) @ W, "y": tr[t]["y"]}, f"{w}/train_frozen_{t}.pt")
                torch.save({"f": (ln(te[t]["f"]) - mu) @ W, "y": te[t]["y"]}, f"{w}/test_frozen_{t}.pt")
            globals()["embed"] = lambda f: F.normalize(f.float(), dim=1)        # đã LayerNorm trước khi chiếu
            r, _ = analyse_seed_src(w, T, seed, "frozen")
            globals()["embed"] = _embed_orig
            for x in r: x.update(dataset=ds, proj=kind, dim=dim)
            ROWS2 += [x for x in r if x["variant"] in ("fuse", "hier")]
    print(f"{ds:13} seed {seed}: xong ({time.time() - t_all:.0f}s)", flush=True)
globals()["embed"] = _embed_orig
pd.DataFrame(ROWS2).to_csv(f"{OUT}/exp2_proj_rows.csv", index=False)


CIFAR100      seed 0: xong (44s)
CIFAR100      seed 1: xong (87s)
CIFAR100      seed 2: xong (131s)
CUB200        seed 0: xong (159s)
CUB200        seed 1: xong (188s)
CUB200        seed 2: xong (216s)
StanfordCars  seed 0: xong (246s)
StanfordCars  seed 1: xong (275s)
StanfordCars  seed 2: xong (306s)
ImageNet_R    seed 0: xong (352s)
ImageNet_R    seed 1: xong (399s)
ImageNet_R    seed 2: xong (443s)


## 6. Bảng kết quả

In [6]:
# ==============================================================================
# 6. BẢNG KẾT QUẢ (chọn cấu hình theo Last trung bình các seed trên Val, như các notebook trước)
# ==============================================================================
CFG = ["w", "kappa", "k", "lam", "w_in"]
fmt = lambda m, s: f"{m:6.2f}" + (f" ± {s:4.2f}" if pd.notna(s) else "       ")

def best(df, keys):
    df = df.copy(); df.loc[(df.variant == "fuse") & (df.w == 1.0), "variant"] = "proto"
    g = df.groupby(keys + ["variant"] + CFG, dropna=False).agg(n=("seed", "nunique"), avg=("avg", "mean"),
        last=("last", "mean"), last_sd=("last", "std"), forg=("forgetting", "mean")).reset_index()
    return g.loc[g.groupby(keys + ["variant"], dropna=False)["last"].idxmax()]

R1 = pd.DataFrame(ROWS1); R1F = pd.DataFrame(ROWS1F); R2 = pd.DataFrame(ROWS2)
OUTS = []
for ds in DATASETS:
    if ds not in R1.dataset.values: continue
    print(f"\n==================== {ds} ====================")
    print("[1] KAC hiệu chỉnh — không nhánh phụ, 1 lượt forward")
    a = R1[R1.dataset == ds].groupby("calib").agg(n=("seed", "nunique"), avg=("avg", "mean"), last=("last", "mean"),
        last_sd=("last", "std"), forg=("forgetting", "mean"), task_id=("task_id_last", "mean")).reindex(CALIB)
    for h, r in a.iterrows():
        print(f"   {h:14} Avg {r.avg:6.2f} | Last {fmt(r['last'], r.last_sd)} | Forg {r.forg:5.2f} | chọn đúng task (sau task cuối) {r.task_id:5.2f}")
    a.insert(0, "dataset", ds); OUTS.append(a.reset_index().assign(table="1_kac_calib"))

    print("[1b] KAC hiệu chỉnh + Spherical-ViT (2 lượt forward như cũ)")
    base = R2[(R2.dataset == ds) & (R2.proj == "full")].assign(kac="raw")
    b = best(pd.concat([base, R1F[R1F.dataset == ds]]), ["kac"])
    for _, r in b.sort_values(["variant", "kac"]).iterrows():
        cfg = " ".join(f"{c}={r[c]:g}" for c in CFG if pd.notna(r[c]))
        print(f"   {r.variant:5} KAC={r.kac:14} Last {fmt(r['last'], r.last_sd)} | Forg {r.forg:5.2f} | {cfg}")
    OUTS.append(b.assign(dataset=ds, table="1b_calib_fuse"))

    print("[2] Giảm chiều Spherical-ViT (KAC gốc)")
    c = best(R2[R2.dataset == ds], ["proj", "dim"])
    piv = c.pivot_table(index=["proj", "dim"], columns="variant", values="last", dropna=False)
    sd = c.pivot_table(index=["proj", "dim"], columns="variant", values="last_sd", dropna=False)
    order = [("full", 768)] + [(p, d) for p in PROJ_TYPES for d in sorted(PROJ_DIMS, reverse=True)]
    for p, d in order:
        if (p, d) not in piv.index: continue
        kk = int(c[(c.proj == p) & (c.dim == d) & (c.variant == "fuse")]["k"].iloc[0])
        print(f"   {p:4} d={d:4} | proto {fmt(piv.loc[(p, d), 'proto'], sd.loc[(p, d), 'proto'])} | fuse {fmt(piv.loc[(p, d), 'fuse'], sd.loc[(p, d), 'fuse'])}"
              f" | hier {fmt(piv.loc[(p, d), 'hier'], sd.loc[(p, d), 'hier'])} | bộ nhớ tâm lớp (fuse, k={kk}) ~{kk * d} số/lớp")
    OUTS.append(c.assign(dataset=ds, table="2_projection"))

pd.concat(OUTS, ignore_index=True).to_csv(f"{OUT}/exp_summary_{PHASE}.csv", index=False, float_format="%.3f")
pd.DataFrame(EVR).groupby(["dataset", "proj", "dim"]).explained_var_task1.mean().dropna().round(3).to_csv(f"{OUT}/pca_explained_var.csv")
print("\nPCA: tỉ lệ phương sai giữ lại (trên ảnh train task 1):")
print(pd.DataFrame(EVR).dropna().groupby(["dataset", "dim"]).explained_var_task1.mean().round(3).unstack())



==================== CIFAR100 ====================
[1] KAC hiệu chỉnh — không nhánh phụ, 1 lượt forward
   raw            Avg  92.33 | Last  88.01 ± 0.72 | Forg  3.65 | chọn đúng task (sau task cuối) 88.45
   center         Avg  92.68 | Last  88.39 ± 0.64 | Forg  3.56 | chọn đúng task (sau task cuối) 88.88
   center_half    Avg  92.57 | Last  88.24 ± 0.62 | Forg  3.61 | chọn đúng task (sau task cuối) 88.72
   zscore         Avg  82.86 | Last  73.22 ± 1.15 | Forg 10.95 | chọn đúng task (sau task cuối) 73.26
   block_softmax  Avg  90.15 | Last  84.78 ± 0.53 | Forg  5.45 | chọn đúng task (sau task cuối) 84.90
[1b] KAC hiệu chỉnh + Spherical-ViT (2 lượt forward như cũ)
   fuse  KAC=block_softmax  Last  85.39 ± 0.61 | Forg  5.97 | w=0.25 kappa=8 k=5
   fuse  KAC=center         Last  89.33 ± 0.52 | Forg  3.35 | w=0.25 kappa=16 k=3
   fuse  KAC=raw            Last  89.08 ± 0.60 | Forg  3.60 | w=0.25 kappa=32 k=5
   hier  KAC=block_softmax  Last  85.01 ± 0.40 | Forg  6.14 | kappa=8 k=5 lam=0.

In [7]:
import zipfile
z = f"/kaggle/working/exp_calib_proj_{PHASE}.zip" if OUT.startswith("/kaggle") else f"{OUT}/results_{PHASE}.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
    for p in glob.glob(f"{OUT}/*.csv"): f.write(p, os.path.basename(p))
shutil.rmtree(f"{OUT}/work", ignore_errors=True)       # thư mục tạm (symlink + logit đã hiệu chỉnh), không cần giữ
print("Tải về:", z)


Tải về: /kaggle/working/exp_calib_proj_val.zip
